# Hugging Face Playground

This notebook demonstrates working with Hugging Face models both via the API and locally. It includes multimodal examples such as text generation, image generation & editing, audio transcription, and (optionally) downloading and transcribing a short video.

## 1) Setup & Install Dependencies

Install required packages for Hugging Face `transformers`, `diffusers`, `whisper`, and utilities like `yt-dlp` for downloading videos.

In [ ]:
# Install required libraries (run once)
# Note: This may take a few minutes.
!pip install -q transformers diffusers accelerate safetensors huggingface_hub yt-dlp torch torchvision torchaudio openai-whisper einops soundfile sentencepiece datasets scipy

## 2) Authenticate with Hugging Face (Optional)

If you want to use the Hugging Face API or access private models, set your `HF_TOKEN` as an environment variable.

> **Tip:** You can create/get a token at https://huggingface.co/settings/tokens.

In [ ]:
import os

# Option 1: Set token in this notebook (not recommended for sharing)
# os.environ["HF_TOKEN"] = "<YOUR_TOKEN>"

# Option 2: Use the huggingface-cli to login once (recommended):
# !huggingface-cli login

# Verify token (optional)
print("HF_TOKEN set?", "HF_TOKEN" in os.environ)

## 3) Text Generation (Hugging Face Inference API)

Demonstrates using the Hugging Face *Inference API* via `InferenceClient` (remote model) for text generation.

> **Note:** The old `api-inference.huggingface.co` endpoint is deprecated (returns 410). The `InferenceClient` from `huggingface_hub` automatically routes through the new `router.huggingface.co` endpoint.

In [ ]:
from huggingface_hub import InferenceClient

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set. Please set it as an environment variable or login with `huggingface-cli login`.")

# InferenceClient uses the new router.huggingface.co endpoint automatically
client = InferenceClient(token=hf_token)

# Text generation using the Inference API
# Using gpt2 which is reliably available on the free tier
prompt = "In a future where AI assistants help people learn every day,"
result = client.text_generation(
    prompt,
    model="gpt2",
    max_new_tokens=120,
    temperature=0.7,
)

print(result)

## 4) Text Generation (Local)

Use the `transformers` pipeline to generate text from a local model (e.g., `gpt2`).

You can swap in any other compatible model from the Hugging Face Hub.

In [ ]:
from transformers import pipeline

# Create a text-generation pipeline (will download model weights the first time)
text_gen = pipeline("text-generation", model="gpt2", device=-1)  # set device=0 for GPU

prompt = "Once upon a time in a world where AI could"
outputs = text_gen(prompt, max_length=120, do_sample=True, temperature=0.8, num_return_sequences=1)

print(outputs[0]["generated_text"])

## 4) Image Generation (Local)

Use `diffusers` to generate an image from a text prompt. This example uses `runwayml/stable-diffusion-v1-5` (or another model of your choice).

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

# Note: runwayml/stable-diffusion-v1-5 has moved to this new repo
model_id = "stable-diffusion-v1-5/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
pipe = pipe.to("cuda" if torch.cuda.is_available() else "cpu")

prompt = "A fantasy landscape with glowing trees, cinematic lighting"
image = pipe(prompt, guidance_scale=7.5).images[0]

# Display in notebook
display(image)

# Save to file
image.save("hf_play_stable_diffusion.png")

## 5) Image Editing (Inpainting)

Use a mask image to edit parts of an existing image. The mask should be white where you want to change the image, and black for areas that should remain unchanged.

In [ ]:
from diffusers import StableDiffusionInpaintPipeline

# Note: runwayml/stable-diffusion-inpainting has moved to this new repo
inpaint_model_id = "stable-diffusion-v1-5/stable-diffusion-inpainting"

inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(inpaint_model_id, torch_dtype=torch.float16)
inpaint_pipe = inpaint_pipe.to("cuda" if torch.cuda.is_available() else "cpu")

# NOTE: Provide your own base image and mask files in the folder.
# from PIL import Image
# base_image = Image.open("path/to/base_image.png").convert("RGB").resize((512, 512))
# mask_image = Image.open("path/to/mask.png").convert("RGB").resize((512, 512))

# prompt = "A rocket ship launching into space"
# result = inpaint_pipe(prompt=prompt, image=base_image, mask_image=mask_image).images[0]
# display(result)
# result.save("hf_play_inpaint.png")

print("Please add your own 'base_image' and 'mask_image' files and uncomment the inpainting lines to run.")

## 6) Audio Transcription (Whisper)

Download an audio file or a short YouTube video (via `yt-dlp`) and transcribe it using `openai-whisper`. This cell shows both steps.

In [ ]:
import subprocess
from pathlib import Path

# Download a short YouTube clip (or use a local file).
# NOTE: Change the URL to a short clip you are allowed to download.
yt_url = "https://www.youtube.com/watch?v=2Vv-BfVoq4g"  # replace with your preferred video
out_path = Path("downloaded_video.mp4")

# Use yt-dlp if available (fall back to youtube-dl if not)
try:
    subprocess.run(["yt-dlp", "-f", "bestaudio[ext=m4a]/bestaudio", "-o", str(out_path), yt_url], check=True)
except FileNotFoundError:
    subprocess.run(["youtube-dl", "-f", "bestaudio[ext=m4a]/bestaudio", "-o", str(out_path), yt_url], check=True)

print(f"Downloaded video to {out_path}")

# Use whisper to transcribe
import whisper

model = whisper.load_model("small")
result = model.transcribe(str(out_path))
print("Transcription (first 500 chars):")
print(result["text"][0:500])


## 7) Optional: Text-to-Speech (TTS)

Generate speech audio from text using a Hugging Face TTS model. This example uses `microsoft/speecht5_tts` which works with the transformers pipeline.

You'll also need speaker embeddings from `Matthijs/cmu-arctic-xvectors`.

In [ ]:
from transformers import pipeline
from datasets import load_dataset
import torch
import soundfile as sf

# Load TTS pipeline with SpeechT5
tts = pipeline("text-to-speech", model="microsoft/speecht5_tts")

# SpeechT5 requires speaker embeddings — load a sample from xvectors dataset
embeddings_dataset = load_dataset("Matthijs/cmu-arctic-xvectors", split="validation")
speaker_embedding = torch.tensor(embeddings_dataset[7306]["xvector"]).unsqueeze(0)

# Generate speech
speech = tts(
    "Hello! This is a demo of text to speech using Hugging Face models.",
    forward_params={"speaker_embeddings": speaker_embedding}
)

# Save to WAV file
output_path = "hf_play_tts.wav"
sf.write(output_path, speech["audio"], samplerate=speech["sampling_rate"])
print(f"Saved TTS output to {output_path}")